# 2. Feature Importance Cohort Runner

This root workflow notebook runs **Step 3a: MC-CV feature importance** for the production cohort matrix.

- **Script**: `3a_feature_importance/run_mc_feature_importance.py`
- **Data location (EC2)**: `/mnt/nvme/{PROJECT_SLUG}/gold/cohorts` (synced from S3 `gold/{PROJECT_SLUG}/cohorts/`)
- **Feature-importance local root (EC2)**: `/mnt/nvme/{PROJECT_SLUG}/3a_feature_importance/outputs`
- **Environment**: Python with `catboost`, `xgboost`, `scikit-learn`, `pandas`, `numpy`

## Cohorts

| Cohort | Target | Notes |
|--------|--------|-------|
| `falls` | `fall_injury_any` | Injury evidence + W00–W19 external fall-cause evidence for the same patient within `CPIC_FALL_TARGET_WINDOW_DAYS` (default 7 days) |
| `ed` | `ed_event` | POS=23 or revenue code 045x/0981 |

## Output Files (per cohort × age_band)

- `{cohort}_{age_band}_aggregated_feature_importance.csv`
- `{cohort}_{age_band}_catboost_feature_importance.csv`
- `{cohort}_{age_band}_xgboost_feature_importance.csv`
- `{cohort}_{age_band}_xgboost_rf_feature_importance.csv`
- S3 location: `s3://pgxdatalake/gold/{PROJECT_SLUG}/feature_importance/{cohort}/{age_band}/`

# Environment

## Production Usage

Run cells in order: **Configuration** → **Run All Cohorts × Age Bands** → **Review outputs** → **Sync Results and Code to S3**.

The production path is idempotent: each cohort/age-band run skips completed S3 outputs, and the final `aws s3 sync` only uploads changed outputs to the project-scoped prefix.

Per-cohort cells are retained for debugging a single cohort/age-band. The production grid is `falls` and `ed` for `65-74` and `75-84` only.

In [1]:
import os
import sys
from pathlib import Path

# Resolve Python binary
PYTHON_BIN = Path(os.environ.get("COHORT_RUNNER_PYTHON", sys.executable))
print(f"[INFO] Python binary: {PYTHON_BIN}")

# Resolve project root
def resolve_project_root() -> Path:
    if "__file__" in globals():
        root = Path(__file__).resolve().parent
        if root.name == "3a_feature_importance":
            return root.parent
        return root
    cwd = Path.cwd()
    if cwd.name == "3a_feature_importance":
        return cwd.parent
    return cwd

PROJECT_ROOT = resolve_project_root()
print(f"[INFO] Project root: {PROJECT_ROOT}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from py_helpers.env_utils import get_data_root
from py_helpers.constants import REQUIRED_COHORTS

DATA_ROOT = get_data_root()
COHORTS = REQUIRED_COHORTS  # {"falls": [...bands...], "ed": [...bands...]}

SCRIPT = PROJECT_ROOT / "3a_feature_importance" / "run_mc_feature_importance.py"
print(f"[INFO] FI script: {SCRIPT}")
print(f"[INFO] Data root: {DATA_ROOT}")
print(f"[INFO] Cohorts: { {k: len(v) for k, v in COHORTS.items()} } (cohort: n_bands)")

[INFO] Python binary: /home/pgx3874/jupyter-env/bin/python3.11
[INFO] Project root: /home/pgx3874/cpic_time_to_event_analysis
[INFO] FI script: /home/pgx3874/cpic_time_to_event_analysis/3a_feature_importance/run_mc_feature_importance.py
[INFO] Data root: /mnt/nvme
[INFO] Cohorts: {'falls': 2, 'ed': 2} (cohort: n_bands)


## Per-Cohort Runner Cells

Each cell below runs Step 3a for a **single cohort × age_band**. All cells use `run_mc_feature_importance.py` and are idempotent (skip when S3 outputs already exist).

Run Configuration (cell above) before running any cell here.

### Falls — Age 65–74 *(primary age band)*

In [ ]:
import subprocess
cohort, age_band = "falls", "65-74"
result = subprocess.run(
    [str(PYTHON_BIN), str(SCRIPT), "--cohort", cohort, "--age_band", age_band],
    cwd=str(PROJECT_ROOT),
)
print(f"{cohort}/{age_band} exit code: {result.returncode}")

[INFO] Writing Step 3 outputs to project-scoped FI root: /mnt/nvme/cpic_time_to_event/3a_feature_importance/outputs/falls
[INFO] Baseline missing in pgx-repository; running baseline (permutation feature importance) first.
[INFO] Writing Step 3 outputs to project-scoped FI root: /mnt/nvme/cpic_time_to_event/3a_feature_importance/outputs/falls
[INFO] Baseline run: writing to _baseline subfolder (original aggregated FI for 1b event filter)
Synced cohort.parquet from S3 to local: /mnt/nvme/cpic_time_to_event/gold/cohorts/cohort_name=falls/event_year=2016/age_band=65-74/cohort.parquet
Synced cohort.parquet from S3 to local: /mnt/nvme/cpic_time_to_event/gold/cohorts/cohort_name=falls/event_year=2017/age_band=65-74/cohort.parquet
Synced cohort.parquet from S3 to local: /mnt/nvme/cpic_time_to_event/gold/cohorts/cohort_name=falls/event_year=2018/age_band=65-74/cohort.parquet
Synced cohort.parquet from S3 to local: /mnt/nvme/cpic_time_to_event/gold/cohorts/cohort_name=falls/event_year=2019/age_b

### Falls — Age 75–84 *(primary age band)*

In [ ]:
import subprocess
cohort, age_band = "falls", "75-84"
result = subprocess.run(
    [str(PYTHON_BIN), str(SCRIPT), "--cohort", cohort, "--age_band", age_band],
    cwd=str(PROJECT_ROOT),
)
print(f"{cohort}/{age_band} exit code: {result.returncode}")

### ED — Age 65–74

In [ ]:
import subprocess
cohort, age_band = "ed", "65-74"
result = subprocess.run(
    [str(PYTHON_BIN), str(SCRIPT), "--cohort", cohort, "--age_band", age_band],
    cwd=str(PROJECT_ROOT),
)
print(f"{cohort}/{age_band} exit code: {result.returncode}")

### ED — Age 75–84

In [ ]:
import subprocess
cohort, age_band = "ed", "75-84"
result = subprocess.run(
    [str(PYTHON_BIN), str(SCRIPT), "--cohort", cohort, "--age_band", age_band],
    cwd=str(PROJECT_ROOT),
)
print(f"{cohort}/{age_band} exit code: {result.returncode}")

## Run All Cohorts × Age Bands (loop)

In [ ]:
## Run All (65-85 group: falls + ed x 65-74, 75-84)

Runs Step 3a for all 4 combinations sequentially. Idempotent - skips when S3 outputs exist.

In [ ]:
import subprocess

FAIL_FAST = True
AGE_BANDS = ["65-74", "75-84"]

for cohort in COHORTS:
    for age_band in COHORTS[cohort]:
        if age_band not in AGE_BANDS:
            continue
        print(f"→ Step 3a: {cohort} / {age_band}")
        r = subprocess.run(
            [str(PYTHON_BIN), str(SCRIPT), "--cohort", cohort, "--age_band", age_band],
            cwd=str(PROJECT_ROOT),
        )
        print(f"  exit code: {r.returncode}")
        if r.returncode != 0 and FAIL_FAST:
            raise RuntimeError(f"Step 3a failed for {cohort}/{age_band}")

print("All Step 3a runs complete.")

# Review: Aggregated feature importance outputs

In [ ]:
from pathlib import Path

OUTPUTS_DIR = PROJECT_ROOT / "3a_feature_importance" / "outputs"
AGE_BANDS = ["65-74", "75-84"]
print(f"Outputs: {OUTPUTS_DIR}\n")
for cohort in COHORTS:
    for age_band in AGE_BANDS:
        ab_f = age_band.replace("-", "_")
        csv = OUTPUTS_DIR / cohort / ab_f / f"{cohort}_{ab_f}_aggregated_feature_importance.csv"
        if csv.exists():
            import pandas as pd
            df = pd.read_csv(csv)
            print(f"  {cohort}/{age_band}: {len(df)} features  ({csv.name})")
        else:
            print(f"  {cohort}/{age_band}: NOT FOUND — run Step 3a first")

In [ ]:
## Sync Results and Code to S3

In [ ]:
## Shutdown EC2 (optional)

In [ ]:
import shutil, subprocess as _sp, os

SHUTDOWN_EC2 = False
if SHUTDOWN_EC2:
    try:
        instance_id = _sp.run(
            ["curl", "-s", "http://169.254.169.254/latest/meta-data/instance-id"],
            capture_output=True, text=True, timeout=5
        ).stdout.strip()
        if instance_id:
            aws_cmd = shutil.which("aws") or "aws"
            _p = ["--profile", os.environ["AWS_PROFILE"]] if os.environ.get("AWS_PROFILE") else []
            r = _sp.run([aws_cmd, "ec2", "stop-instances", "--instance-ids", instance_id] + _p,
                        capture_output=True, text=True)
            print(f"EC2 stop {instance_id}: {'OK' if r.returncode == 0 else r.stderr.strip()}")
    except Exception as e:
        print(f"Shutdown skipped: {e}")
else:
    print("EC2 auto-shutdown DISABLED")

# Sync Results and Code to S3

Sync Step 3a output files to the project-scoped S3 prefix only:

`s3://{S3_BUCKET}/gold/{PROJECT_SLUG}/feature_importance/`

In [ ]:
import os
import shutil
import subprocess

# Sync 3a outputs to the project-scoped S3 prefix (idempotent).
S3_BUCKET = os.environ.get("CPIC_S3_BUCKET", "pgxdatalake")
PROJECT_SLUG = os.environ.get("CPIC_PROJECT_SLUG", "cpic_time_to_event")
S3_FI_PREFIX = f"s3://{S3_BUCKET}/gold/{PROJECT_SLUG}/feature_importance/"
OUTPUTS_DIR = PROJECT_ROOT / "3a_feature_importance" / "outputs"
_aws = shutil.which("aws") or "aws"
AWS_PROFILE = os.environ.get("AWS_PROFILE")
_profile = ["--profile", AWS_PROFILE] if AWS_PROFILE else []

if OUTPUTS_DIR.exists():
    r = subprocess.run(
        [_aws, "s3", "sync", str(OUTPUTS_DIR), S3_FI_PREFIX] + _profile,
        capture_output=False,
    )
    print(f"S3 sync exit code: {r.returncode}")
else:
    print(f"Outputs dir not found: {OUTPUTS_DIR}")

# Shutdown EC2

In [ ]:
import shutil, subprocess as _sp, os

SHUTDOWN_EC2 = False  # Set True when running on EC2

if SHUTDOWN_EC2:
    try:
        instance_id = _sp.run(
            ["curl", "-s", "http://169.254.169.254/latest/meta-data/instance-id"],
            capture_output=True, text=True, timeout=5
        ).stdout.strip()
        if instance_id:
            aws_cmd = shutil.which("aws") or "aws"
            _profile = ["--profile", os.environ["AWS_PROFILE"]] if os.environ.get("AWS_PROFILE") else []
            r = _sp.run([aws_cmd, "ec2", "stop-instances", "--instance-ids", instance_id] + _profile,
                        capture_output=True, text=True)
            print(f"EC2 stop {instance_id}: {'OK' if r.returncode == 0 else r.stderr.strip()}")
        else:
            print("EC2 instance ID not found - are you on EC2?")
    except Exception as e:
        print(f"EC2 shutdown skipped: {e}")
else:
    print("EC2 auto-shutdown DISABLED (SHUTDOWN_EC2 = False)")